# Découpage généralisé de toutes les couches vectorielles

Ce notebook vérifie que `processing.clipping.clip_all_vector_layers` découpe automatiquement **toutes** les couches vectorielles déjà téléchargées (`data/raw/vector/`) selon la limite communale officielle + un buffer, et les écrit en miroir dans `data/processed/vector/`.

In [ ]:
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

from core.config import load_entities
from download.limites_administratives import fetch_commune_boundary
from processing.clipping import clip_all_vector_layers

## 1. Entité testée (issue du CSV)

In [ ]:
csv_path = repo_root / "data" / "configuration" / "entites_exemple.csv"
entity = load_entities(csv_path)[0]
entity

## 2. Découpage de toutes les couches (limite communale + buffer 50m)

In [ ]:
boundary = fetch_commune_boundary(entity.code_insee)

raw_dir = repo_root / "data" / "raw" / "vector"
processed_dir = repo_root / "data" / "processed" / "vector"

written = clip_all_vector_layers(raw_dir, processed_dir, boundary, buffer_distance=50)

print("Couches decoupees :", len(written))
for rel, path in written.items():
    print(" -", rel)

## 3. Affichage des couches traitées

Reprojection en EPSG:4326 uniquement pour l'affichage. Backend `folium`, plus fiable pour le rendu des widgets dans VS Code.

In [ ]:
import geopandas as gpd
import leafmap.foliumap as leafmap

boundary_wgs84 = boundary.to_crs(epsg=4326)

m = leafmap.Map()

for rel, path in written.items():
    gdf = gpd.read_file(path)
    if gdf.empty:
        continue
    m.add_gdf(gdf.to_crs(epsg=4326), layer_name=rel, info_mode="on_click")

m.add_gdf(
    boundary_wgs84,
    layer_name="Limite communale",
    style={"color": "red", "fillOpacity": 0},
)
m.zoom_to_gdf(boundary_wgs84)
m